# 13.07 - Macro-F1 + Imbalance

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Per-class CV error report.

Today is about reading model quality through the competition metric, not just through accuracy. In imbalanced computer-vision tasks, a model can look strong overall while quietly failing the rare classes that matter most for Macro-F1.


## Core Ideas

Accuracy counts every image equally, so majority classes can dominate the score. Macro-F1 gives each class equal influence: compute F1 for each class, then average those class scores.

For one class treated as the positive class:

- `precision = true_positives / predicted_positives`
- `recall = true_positives / actual_positives`
- `f1 = 2 * precision * recall / (precision + recall)`

The confusion matrix is the map behind those numbers. Rows are true labels, columns are predicted labels. False negatives for a class live across that class row, and false positives for a class live down that class column.

When a dataset is imbalanced, two common training ideas are:

- **Weighted loss:** make mistakes on rare classes cost more.
- **Weighted sampling:** draw rare-class examples more often during training batches.

Both are tools, not magic. Use per-class validation metrics to verify whether they actually help.


In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

LABELS = ["clear", "crack", "scratch", "stain"]


## Prepared CV Prediction Data

The table below simulates validation predictions for an imbalanced image-classification problem. The majority class is `clear`; rare defect classes are intentionally harder.

The provided data cell is complete. Run it before the exercises.


In [11]:
def make_imbalanced_cv_predictions(labels=LABELS):
    true_labels = (
        ["clear"] * 24
        + ["crack"] * 8
        + ["scratch"] * 5
        + ["stain"] * 3
    )

    predicted_labels = (
        ["clear"] * 22 + ["scratch", "crack"]
        + ["crack"] * 5 + ["clear", "clear", "scratch"]
        + ["scratch", "scratch", "clear", "clear", "crack"]
        + ["clear", "stain", "scratch"]
    )

    confidences = np.array([
        0.97, 0.94, 0.91, 0.88, 0.86, 0.93, 0.82, 0.79,
        0.96, 0.91, 0.89, 0.87, 0.85, 0.81, 0.77, 0.74,
        0.92, 0.90, 0.84, 0.80, 0.76, 0.73, 0.61, 0.66,
        0.78, 0.71, 0.69, 0.64, 0.58, 0.72, 0.67, 0.55,
        0.82, 0.76, 0.57, 0.60, 0.62, 0.74, 0.63, 0.52,
    ])

    rows = []
    for idx, (true_label, predicted_label, confidence) in enumerate(zip(true_labels, predicted_labels, confidences)):
        rows.append({
            "image_id": "val_%03d.png" % idx,
            "true_label": true_label,
            "pred_label": predicted_label,
            "confidence": float(confidence),
            "is_correct": true_label == predicted_label,
        })
    return pd.DataFrame(rows)

val_predictions = make_imbalanced_cv_predictions()
val_predictions.head()


,image_id,true_label,pred_label,confidence,is_correct
0,val_000.png,clear,clear,0.97,True
1,val_001.png,clear,clear,0.94,True
2,val_002.png,clear,clear,0.91,True
3,val_003.png,clear,clear,0.88,True
4,val_004.png,clear,clear,0.86,True


## Exercise 13-A: Imbalance Summary

Write `summarize_imbalance(df, labels)`.

Return a `DataFrame` indexed by label with columns:

- `support`: number of validation examples for the class
- `proportion`: class share of the validation set
- `majority_ratio`: largest class count divided by this class count

The rarest class should have the largest `majority_ratio`.


In [18]:
# TODO 13-A
def summarize_imbalance(df, labels):
    counts = df["true_label"].value_counts().reindex(labels, fill_value = 0)
    total_samples = counts.sum()
    highest_class = counts.max()
    summary = pd.DataFrame(index = labels)
    summary["support"] = counts.values
    summary["proportion"] = summary["support"]/total_samples
    summary["majority_ratio"] = highest_class/counts.replace(0,np.nan).values
    return summary


## Exercise 13-B: Confusion Matrix and Per-Class Metrics

Write two functions:

- `build_confusion_df(y_true, y_pred, labels)` returns a labeled confusion matrix as a `DataFrame`.
- `compute_per_class_metrics(y_true, y_pred, labels)` returns a `DataFrame` indexed by label with `precision`, `recall`, `f1`, and `support`.

Use zero when precision, recall, or F1 would divide by zero.


In [44]:
# TODO 13-B
def build_confusion_df(y_true, y_pred, labels):
    matrix = confusion_matrix(y_true,y_pred,labels = labels)
    return pd.DataFrame(matrix, index = labels, columns = labels)


def compute_per_class_metrics(y_true, y_pred, labels):
    precision, recall, f1, support = precision_recall_fscore_support(y_true,y_pred,labels = labels,zero_division = 0)
    metrics = pd.DataFrame({
        "precision" : precision,
        "recall" : recall,
        "f1" : f1,
        "support" : support
    }, index = labels)
    return metrics

## Exercise 13-C: Class Weights for Weighted Loss

Write `make_class_weights(y_true, labels)` using the common balanced-weight formula:

`weight[class] = n_samples / (n_classes * class_count)`

Return a dictionary mapping label name to float weight. Rare classes should receive larger weights.


In [32]:
# TODO 13-C
def make_class_weights(y_true, labels):
    n_samples = len(y_true)
    n_classes = len(labels)
    counts = y_true.value_counts()
    weight = {}
    for label, counts in counts.items() : 
        if counts == 0 : weight[label] = 0
        else : weight[label] = n_samples/(n_classes * counts)
    return weight


## Exercise 13-D: Weighted Loss Diagnostic

Write `weighted_loss_diagnostic(logits, y_true, labels, class_weights)`.

The function should:

1. Convert string labels into class indices using `labels`.
2. Compute unweighted per-example cross-entropy losses.
3. Compute weighted per-example cross-entropy losses using `class_weights`.
4. Return a `DataFrame` with one row per class and columns `unweighted_loss`, `weighted_loss`, and `weight`.

This is a diagnostic: it shows how weighting changes the average loss contribution by class.


In [38]:
# TODO 13-D
def weighted_loss_diagnostic(logits, y_true, labels, class_weights):
    label_to_idx = {label: idx for idx, label in enumerate(labels)}
    target = torch.tensor([label_to_idx[i] for i in y_true])
    weights = torch.tensor([class_weights[i] for i in labels])
    unweighted = torch.nn.functional.cross_entropy(logits,target,reduction = "none")
    weighted = torch.nn.functional.cross_entropy(logits,target,weight = weights,reduction = "none")
    rows = []
    for label in labels : 
        idx = label_to_idx[label]
        mask = target == idx
        rows.append({
            "label" : label,
            "unweighted_loss" : unweighted[mask].mean().item(),
            "weighted_loss" : weighted[mask].mean().item(),
            "weight" : class_weights[label]
        })
    return pd.DataFrame(rows).set_index("label")



## Exercise 13-E: Sampler Weights and Error Report

Write two functions:

- `make_sample_weights(y_true, class_weights)` returns a NumPy array with one weight per example.
- `make_error_report(df, labels)` returns a `DataFrame` indexed by label with `support`, `precision`, `recall`, `f1`, `false_positives`, `false_negatives`, and `most_confused_as`.

`most_confused_as` should be the wrong predicted label most often used for that true class. If a class has no false negatives, use `None`.


In [50]:
# TODO 13-E
def make_sample_weights(y_true, class_weights):
    sample_weights = np.array([class_weights[i] for i in y_true], dtype = np.float32)
    return sample_weights

def make_error_report(df, labels):
    confusion_matrix = build_confusion_df(df["true_label"],df["pred_label"],labels)
    metrics = compute_per_class_metrics(df["true_label"],df["pred_label"],labels)

    FP = []
    FN = []
    AS = []

    for label in labels : 
        tp = int(confusion_matrix.loc[label,label])
        fp = int(confusion_matrix[label].sum() - tp)
        fn = int(confusion_matrix.loc[label].sum() - tp)
        FP.append(fp)
        FN.append(fn)

        row = confusion_matrix.loc[label].copy()
        row.loc[label] = 0
        if int(row.sum()) == 0 :
            AS.append(None)
        else : AS.append(row.idxmax())
    
    metrics["false_positives"] = FP
    metrics["false_negatives"] = FN
    metrics["most_confused_as"] = AS
    return metrics[["support","precision","recall","f1","false_positives","false_negatives","most_confused_as"]]
    


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 13 tests passed`.


In [51]:
def run_day13_tests():
    assert "summarize_imbalance" in globals(), "Missing function: summarize_imbalance"
    assert "build_confusion_df" in globals(), "Missing function: build_confusion_df"
    assert "compute_per_class_metrics" in globals(), "Missing function: compute_per_class_metrics"
    assert "make_class_weights" in globals(), "Missing function: make_class_weights"
    assert "weighted_loss_diagnostic" in globals(), "Missing function: weighted_loss_diagnostic"
    assert "make_sample_weights" in globals(), "Missing function: make_sample_weights"
    assert "make_error_report" in globals(), "Missing function: make_error_report"

    df = make_imbalanced_cv_predictions()
    y_true = df["true_label"]
    y_pred = df["pred_label"]

    summary = summarize_imbalance(df, LABELS)
    assert list(summary.index) == LABELS
    assert summary.loc["clear", "support"] == 24
    assert summary.loc["stain", "support"] == 3
    assert np.isclose(summary["proportion"].sum(), 1.0)
    assert summary.loc["stain", "majority_ratio"] == 8.0

    confusion = build_confusion_df(y_true, y_pred, LABELS)
    expected_confusion = confusion_matrix(y_true, y_pred, labels=LABELS)
    assert confusion.shape == (4, 4)
    assert np.array_equal(confusion.values, expected_confusion)
    assert int(confusion.loc["clear", "clear"]) == 22
    assert int(confusion.loc["stain", "stain"]) == 1

    metrics = compute_per_class_metrics(y_true, y_pred, LABELS)
    ref_precision, ref_recall, ref_f1, ref_support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    assert np.allclose(metrics["precision"].values, ref_precision)
    assert np.allclose(metrics["recall"].values, ref_recall)
    assert np.allclose(metrics["f1"].values, ref_f1)
    assert np.array_equal(metrics["support"].values, ref_support)
    assert np.isclose(metrics["f1"].mean(), 0.6073529411764705)

    weights = make_class_weights(y_true, LABELS)
    assert weights["clear"] < weights["crack"] < weights["scratch"] < weights["stain"]
    assert np.isclose(weights["clear"], 40 / (4 * 24))
    assert np.isclose(weights["stain"], 40 / (4 * 3))

    logits = torch.tensor([
        [2.9, 0.3, 0.1, -0.4],
        [2.6, 0.1, 0.2, -0.2],
        [0.6, 2.1, 0.4, 0.0],
        [1.7, 0.8, 0.3, 0.1],
        [0.4, 0.2, 2.0, 0.2],
        [1.4, 0.4, 0.9, 0.1],
        [1.2, 0.1, 0.4, 1.1],
        [0.5, 0.1, 0.7, 1.8],
    ], dtype=torch.float32)
    mini_true = ["clear", "clear", "crack", "crack", "scratch", "scratch", "stain", "stain"]
    loss_report = weighted_loss_diagnostic(logits, mini_true, LABELS, weights)
    assert list(loss_report.index) == LABELS
    assert set(loss_report.columns) == {"unweighted_loss", "weighted_loss", "weight"}
    assert np.isclose(loss_report.loc["stain", "weight"], weights["stain"])
    assert loss_report.loc["stain", "weighted_loss"] > loss_report.loc["stain", "unweighted_loss"]
    assert loss_report.loc["clear", "weighted_loss"] < loss_report.loc["clear", "unweighted_loss"]

    sample_weights = make_sample_weights(y_true, weights)
    assert isinstance(sample_weights, np.ndarray)
    assert sample_weights.shape == (len(df),)
    assert sample_weights.dtype == np.float32
    assert np.isclose(sample_weights[y_true == "stain"].mean(), weights["stain"])
    assert sample_weights[y_true == "stain"].mean() > sample_weights[y_true == "clear"].mean()

    report = make_error_report(df, LABELS)
    assert list(report.index) == LABELS
    assert list(report.columns) == ["support", "precision", "recall", "f1", "false_positives", "false_negatives", "most_confused_as"]
    assert int(report.loc["crack", "false_negatives"]) == 3
    assert int(report.loc["clear", "false_positives"]) == 5
    assert report.loc["stain", "most_confused_as"] in ["clear", "scratch"]
    assert np.isclose(report["f1"].mean(), metrics["f1"].mean())

    print("Day 13 tests passed")

run_day13_tests()


Day 13 tests passed


## Day 13 Checklist

- Explain why Macro-F1 can be more contest-relevant than accuracy on imbalanced data.
- Read a confusion matrix by row and by column.
- Compute per-class precision, recall, F1, and Macro-F1.
- Create class weights and sample weights from class frequencies.
- Produce a per-class CV error report that suggests what to fix next.
